# Notebook 11c — Prueba de estrés del NER: redacciones no anticipadas

**Proyecto BME513 · Universidad de Valparaíso** — Sebastián Inostroza Hurtado

---

## Por qué hago este experimento

El NER existe como **respaldo para lo que las reglas no anticiparon**. Su rol vive, por definición, **fuera del corpus de entrenamiento**.

Eso crea un problema de medición que hay que nombrar con precisión:

| | Dónde actúa | ¿La métrica del corpus lo evalúa? |
|---|---|---|
| Verificador BI-RADS (descartado) | **Dentro** del corpus | Sí. Y mostró que no aportaba |
| **NER de recomendación** | **Fuera** del corpus | **No.** Mide otra cosa |

El F1 = 0,9991 solo prueba que el NER aprendió la tarea. **No lo justifica.** De hecho, medido dentro del corpus, un regex trivial lo empata:

```
Sobre el corpus, sin encabezado:
  Regex "desde el ultimo verbo gatillo hasta el final" : F1 = 0,9991
  NER DistilBETO                                        : F1 = 1,0000
```

La justificación real del NER son los informes chilenos, donde las reglas por encabezado fallaron. **Pero son 3 informes.** Es poca evidencia.

## Qué mido aquí

No tengo más informes chilenos. Así que **simulo la brecha sobre los 4 357**: perturbo el corpus para que se parezca a lo que las reglas no anticiparon, y comparo la regla real contra el NER.

| Condición | Perturbación | Qué simula |
|---|---|---|
| **A** | Ninguna (control) | El corpus tal cual |
| **B** | Sin el encabezado `RECOMENDACIONES:` | El informe chileno sin encabezado |
| **C** | Verbo gatillo **fuera de la lista de reglas** | La redacción no anticipada |
| **D** | Ambas | El caso chileno completo |

## Lo que ya se midió sobre la regla (antes de correr esto)

La regla se probó en su **vía 2**: la que corre cuando no hay columna `Recommendations`, o sea la que se ejecuta con un PDF chileno. Sobre 500 informes:

```
A · control                      100.0%
B · sin encabezado                92.4%    <- aguanta
C · verbo fuera de la lista        1.4%    <- colapsa
D · ambas                          1.0%    <- colapsa
```

**La regla no depende del encabezado: depende del verbo.** Sin encabezado sobrevive con 92,4 %, porque el verbo gatillo la salva. Con un verbo que nadie anticipó, cae a 1,4 %.

Ese es el hueco exacto que el NER debería tapar, y es lo que mide este notebook.

## Por qué la condición C es la que importa

Las reglas usan una **lista cerrada** de verbos (`_FRASES_GATILLO_RECOMENDACION`). Es extensa (se sugiere, se recomienda, amerita, procede, realizar, efectuar, completar, y decenas más), pero es cerrada: solo encuentra lo que alguien anticipó.

Los verbos de reemplazo se **verificaron contra el regex real** del módulo: ninguno dispara.

```
'se plantea'      -> LIBRE (el regex no la reconoce)
'se propone'      -> LIBRE
'cabe practicar'  -> LIBRE
...
```

Todos son de **2 tokens**, igual que `"se sugiere"`, así que el span de verdad no se corre ni un token.

**La regla va a fallar en C por construcción, no por casualidad.** Esa es justamente la hipótesis: su cobertura depende de anticipar.

## Hipótesis

| | Resultado en C y D | Lectura |
|---|---|---|
| **H1** | El NER aguanta | Aprendió qué **es** una recomendación, no una lista de verbos. Su rol de respaldo queda medido sobre 4 357 casos, no sobre 3 |
| **H2** | El NER también cae | Aprendió la lista, igual que las reglas. Su justificación vuelve a los 3 informes chilenos, y hay que decirlo |

## Limitación, y va declarada en el resultado

Esto es **sintético**. No es un informe chileno real: es el corpus paraguayo perturbado. Mide el rol que el componente cumple (manejar redacciones no anticipadas), no su desempeño en Chile. La validación con informes chilenos reales sigue siendo el trabajo pendiente.

> **Este notebook no reentrena nada ni modifica archivos.** Carga el modelo guardado y lo evalúa cuatro veces.


---
## Paso 1 — Configuración

Las funciones se **copian textuales del nb 11**. Una versión anterior de esta serie de notebooks dio F1 = 0 durante tres corridas por reescribirlas de memoria.

In [ ]:
import os, re, sys, json, random, warnings, time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification

warnings.filterwarnings("ignore")

MODEL_DIR  = "../models/ner_recomendacion_final"
DATA_PATH  = "../data/processed/reports_cleaned.csv"
SRC_PATH   = ".."                       # para importar src.extractor_recomendacion
MAX_LEN    = 384
SEED       = 42
label_list = ["O", "B-REC", "I-REC"]
F1_NB11    = 0.9991

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")

for cand in [MODEL_DIR, "models/ner_recomendacion_final", "../../models/ner_recomendacion_final"]:
    if Path(cand).exists(): MODEL_DIR = cand; break
for cand in [DATA_PATH, "data/processed/reports_cleaned.csv", "data/reports_cleaned.csv"]:
    if Path(cand).exists(): DATA_PATH = cand; break
assert Path(MODEL_DIR).exists(), f"No encuentro el modelo en {MODEL_DIR}"
assert Path(DATA_PATH).exists(), f"No encuentro el CSV en {DATA_PATH}"

print(f"Device: {DEVICE}")
print(f"Modelo: {MODEL_DIR}")
print(f"Datos : {DATA_PATH}")

---
## Paso 2 — Importo la regla real

No una reimplementación: **el módulo que corre en producción**. Si el regex de gatillos cambia, este experimento cambia con él.

In [ ]:
for p in [SRC_PATH, ".", "..", "../.."]:
    if Path(p, "src", "extractor_recomendacion.py").exists():
        sys.path.insert(0, p); break

from src.extractor_recomendacion import _FRASES_GATILLO_RECOMENDACION as PAT_GATILLO
from src.extractor_recomendacion import extraer_texto_recomendacion as EXTRAER

print(f"Regex de gatillos cargado desde el modulo real ({len(PAT_GATILLO.pattern)} caracteres)")

# Verbos de reemplazo: se comprueba contra el regex REAL que ninguno dispare.
CANDIDATOS = [
    "se plantea", "se propone", "cabe practicar", "resulta pertinente",
    "estimamos prudente", "juzgamos oportuno", "corresponde gestionar",
    "cabe agendar", "se contempla", "se estipula",
]
REEMPLAZOS = []
print("\nVerificando que los reemplazos NO esten en la lista de reglas:")
for c in CANDIDATOS:
    hit = PAT_GATILLO.search(c)
    print(f"   {c:24s} {'DISPARA -> descartado' if hit else 'LIBRE'}")
    if not hit and len(c.split()) == 2:
        REEMPLAZOS.append(c)

assert len(REEMPLAZOS) >= 5, "Muy pocos reemplazos libres"
print(f"\nUso {len(REEMPLAZOS)} reemplazos, todos de 2 tokens (el span no se corre).")

---
## Paso 3 — Reconstruyo el test del nb 11

Funciones **copiadas textuales** del nb 11.

In [ ]:
df = pd.read_csv(DATA_PATH)
assert "Full_Report_clean" in df.columns and "Recommendations_clean" in df.columns, \
    "Faltan las columnas _clean: el nb 11 entreno con ellas"

def limpiar_rec(rec):
    """COPIADA TEXTUAL del nb 11. No reescribir: quita el guion inicial."""
    return str(rec).strip().lstrip("-*\u2022 ").strip()

def etiquetar_bio(full_report, recomendacion):
    """COPIADA TEXTUAL del nb 11."""
    full = str(full_report); rec = limpiar_rec(recomendacion)
    tokens = full.split(); etiquetas = ["O"] * len(tokens)
    rec_tokens = rec.split()
    if not rec_tokens: return tokens, etiquetas
    n, m = len(tokens), len(rec_tokens)
    for i in range(n - m + 1):
        if all(tokens[i+j].strip(".,;:") == rec_tokens[j].strip(".,;:") for j in range(m)):
            etiquetas[i] = "B-REC"
            for j in range(1, m): etiquetas[i+j] = "I-REC"
            break
    return tokens, etiquetas

def firma(tokens):
    """COPIADA TEXTUAL del nb 11."""
    t = " ".join(tokens).lower()
    t = re.sub(r"\d+", "#", t)
    t = re.sub(r"[^a-zñáéíóú# ]", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    return t

datos = []
for _, row in df.iterrows():
    toks, labs = etiquetar_bio(row["Full_Report_clean"], row["Recommendations_clean"])
    if "B-REC" in labs:
        datos.append({"tokens": toks, "labels": labs})

visto, dedup = set(), []
for d in datos:
    f = firma(d["tokens"])
    if f not in visto: visto.add(f); dedup.append(d)

tr, tmp = train_test_split(dedup, test_size=0.30, random_state=SEED, shuffle=True)
va, te  = train_test_split(tmp,  test_size=0.50, random_state=SEED, shuffle=True)
print(f"Con span: {len(datos)} -> dedup {len(dedup)} -> TEST {len(te)}")

---
## Paso 4 — Las cuatro condiciones

**Regla de oro: las etiquetas nunca se tocan.** El span sigue en la misma posición y con el mismo largo. Lo único que cambia es el texto.

In [ ]:
PAT_ENC = re.compile(r"recomendacion(?:es)?\s*:?", re.IGNORECASE)

def quitar_encabezado(tokens, labels):
    nt, nl = [], []
    for t, l in zip(tokens, labels):
        if PAT_ENC.fullmatch(t.strip(":-. ")): continue
        nt.append(t); nl.append(l)
    return nt, nl

def cambiar_verbo(tokens, labels, rng):
    """Reemplaza los 2 primeros tokens del span por un verbo fuera de la lista.

    Se exige 2 tokens exactos para que el span conserve inicio y largo.
    """
    ini = next((i for i, l in enumerate(labels) if l == "B-REC"), None)
    if ini is None or ini + 1 >= len(tokens): return tokens, labels, False
    nuevo = rng.choice(REEMPLAZOS).split()
    t2 = list(tokens)
    t2[ini], t2[ini+1] = nuevo[0], nuevo[1]
    return t2, labels, True

rng = random.Random(SEED)
cond = {}
cond["A"] = [dict(d) for d in te]

cond["B"] = []
for d in te:
    t, l = quitar_encabezado(d["tokens"], d["labels"])
    cond["B"].append({"tokens": t, "labels": l})

cond["C"], n_c = [], 0
for d in te:
    t, l, ok = cambiar_verbo(d["tokens"], d["labels"], rng)
    n_c += ok
    cond["C"].append({"tokens": t, "labels": l})

cond["D"], n_d = [], 0
for d in cond["B"]:
    t, l, ok = cambiar_verbo(d["tokens"], d["labels"], rng)
    n_d += ok
    cond["D"].append({"tokens": t, "labels": l})

print(f"A control        : {len(cond['A'])}")
print(f"B sin encabezado : {len(cond['B'])}")
print(f"C verbo nuevo    : {len(cond['C'])}  ({n_c} con verbo reemplazado)")
print(f"D ambas          : {len(cond['D'])}  ({n_d} con verbo reemplazado)")

def span_de(labels):
    ini = next((i for i, l in enumerate(labels) if l == "B-REC"), None)
    if ini is None: return None
    fin = ini
    while fin + 1 < len(labels) and labels[fin+1] == "I-REC": fin += 1
    return (ini, fin)

# Los spans deben ser identicos en las 4 condiciones (salvo el corrimiento del encabezado)
for k in ["A", "C"]:
    largos = [span_de(d["labels"])[1] - span_de(d["labels"])[0] for d in cond[k] if span_de(d["labels"])]
    ref    = [span_de(d["labels"])[1] - span_de(d["labels"])[0] for d in cond["A"] if span_de(d["labels"])]
    assert largos == ref, f"La condicion {k} altero el largo de los spans"
print("\nOK: los spans conservan su largo. Solo cambio el texto.")

---
## ✋ Checkpoint 1 — Ver un ejemplo antes de gastar cómputo

In [ ]:
i = 0
for k in ["A", "B", "C", "D"]:
    d = cond[k][i]
    s = span_de(d["labels"])
    ctx = " ".join(d["tokens"][max(0, s[0]-6): s[0]+8])
    print(f"  {k}: ...{ctx}...")
print()
print("  A y B deben empezar el span en 'se sugiere' (o similar).")
print("  C y D deben empezar el span en un verbo que las reglas NO conocen.")

---
## Paso 5 — Cargo el modelo

**Ojo con el mapeo de etiquetas.** El nb 11 creó el modelo sin pasar `id2label`, así que HuggingFace le puso `LABEL_0/1/2`. No se puede confiar en `config.id2label`: el orden real es el de `label_list`.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
modelo    = AutoModelForTokenClassification.from_pretrained(MODEL_DIR).to(DEVICE)
modelo.eval()

cfg = {int(k): v for k, v in modelo.config.id2label.items()}
print(f"id2label guardado en el modelo: {cfg}")
if set(cfg.values()) == set(label_list):
    ID2LABEL = cfg
    print("  El modelo trae las etiquetas reales.")
else:
    ID2LABEL = {i: l for i, l in enumerate(label_list)}
    print(f"  El modelo trae etiquetas genericas. Uso el orden del entrenamiento: {ID2LABEL}")
assert set(ID2LABEL.values()) == set(label_list)
print(f"  Mapeo en uso: {ID2LABEL}")

---
## Paso 6 — Los dos métodos a comparar

**La regla** que se evalúa aquí es una reimplementación mínima del criterio del módulo (localizar desde el gatillo hasta el final), usando el **regex real** importado del código de producción. Se compara contra el NER en igualdad de condiciones: ambos reciben exactamente el mismo texto.

In [ ]:
@torch.no_grad()
def predecir_ner(tokens):
    enc = tokenizer(tokens, is_split_into_words=True, truncation=True,
                    max_length=MAX_LEN, return_tensors="pt").to(DEVICE)
    pred_sub = modelo(**enc).logits[0].argmax(-1).cpu().numpy()
    word_ids = enc.word_ids(0)
    out, visto = [], set()
    for sub_i, w_i in enumerate(word_ids):
        if w_i is None or w_i in visto: continue
        visto.add(w_i)
        out.append(ID2LABEL[int(pred_sub[sub_i])])
    while len(out) < len(tokens): out.append("O")
    return out[:len(tokens)]

def predecir_regla_real(tokens):
    """Llama al EXTRACTOR REAL en su via 2 (sin columna Recommendations).

    Es la via que corre con un PDF chileno: no hay columna, hay que buscar
    en el informe. Se le pasa el mismo texto que al NER.
    """
    full = " ".join(tokens)
    out = EXTRAER(None, full_report=full, usar_ner=False)
    texto = str(out.get("texto") or "").strip()
    if not texto: return ["O"] * len(tokens)

    # Alinear el texto devuelto con los indices de token
    rt = limpiar_rec(texto).split()
    pred = ["O"] * len(tokens)
    if not rt: return pred
    n, m = len(tokens), len(rt)
    for i in range(n - m + 1):
        if all(tokens[i+j].strip(".,;:") == rt[j].strip(".,;:") for j in range(m)):
            pred[i] = "B-REC"
            for j in range(1, m): pred[i+j] = "I-REC"
            break
    return pred

def f1_span(conjunto, predictor):
    """F1 de span exacto: mismo inicio y mismo fin. Es lo que hace seqeval."""
    tp = fp = fn = 0
    for d in conjunto:
        sv = span_de(d["labels"])
        sp = span_de(predictor(d["tokens"]))
        if sv is None: continue
        if sp is None:  fn += 1
        elif sv == sp:  tp += 1
        else:           fp += 1; fn += 1
    P = tp / max(tp + fp, 1); R = tp / max(tp + fn, 1)
    return (2 * P * R / max(P + R, 1e-9)), P, R

---
## ✋ Checkpoint 2 — Prueba de humo (5 segundos)

Si el control no da cerca de 1,0 sobre 5 informes, hay un bug y el experimento completo va a fallar igual. Se detiene aquí.

In [ ]:
f, p, r = f1_span(cond["A"][:5], predecir_ner)
print("="*58)
print("PRUEBA DE HUMO — NER sobre 5 informes del control")
print("="*58)
print(f"  F1 = {f:.4f}")
if f < 0.5:
    d = cond["A"][0]
    txt = " ".join(d["tokens"])
    print("\n  FALLA. Diagnostico:")
    print(f"   Texto      : {txt[:90]}...")
    print(f"   Minusculas : {txt == txt.lower()}  ·  sin tildes: {not any(c in txt for c in 'áéíóúñ')}")
    print(f"   Verdad     : {[l for l in d['labels'] if l != 'O'][:5]}")
    print(f"   Predicho   : {[l for l in predecir_ner(d['tokens']) if l != 'O'][:5]}")
    print(f"   Mapeo      : {ID2LABEL}")
    raise RuntimeError("Prueba de humo fallida.")
print("  OK. Sigo.")

---
## Paso 7 — Evaluación de las cuatro condiciones

In [ ]:
NOMBRES = {"A": "A · control (corpus tal cual)",
           "B": "B · sin encabezado RECOMENDACIONES",
           "C": "C · verbo fuera de la lista de reglas",
           "D": "D · sin encabezado Y verbo nuevo"}

res = {}
for k in ["A", "B", "C", "D"]:
    t0 = time.time()
    f_ner, p_ner, r_ner = f1_span(cond[k], predecir_ner)
    f_reg, p_reg, r_reg = f1_span(cond[k], predecir_regla_real)
    res[k] = {"ner": f_ner, "regla": f_reg,
              "ner_p": p_ner, "ner_r": r_ner,
              "regla_p": p_reg, "regla_r": r_reg,
              "minutos": (time.time()-t0)/60}
    print(f"{NOMBRES[k]:42s}  regla {f_reg:.4f}  ·  NER {f_ner:.4f}   ({res[k]['minutos']:.1f} min)")

---
## Paso 8 — El veredicto

In [ ]:
CONTROL_OK = abs(res["A"]["ner"] - F1_NB11) <= 0.02

print("="*70)
print("PRUEBA DE ESTRÉS DEL NER — RESULTADOS")
print("="*70)
print(f"{'Condición':<40} {'Regla':>9} {'NER':>9}")
print("-"*70)
for k in ["A", "B", "C", "D"]:
    print(f"{NOMBRES[k]:<40} {res[k]['regla']:>9.4f} {res[k]['ner']:>9.4f}")

print()
if not CONTROL_OK:
    veredicto = "EXPERIMENTO INVÁLIDO"
    print("="*70)
    print("EXPERIMENTO INVÁLIDO. NO LEAS EL RESTO.")
    print("="*70)
    print(f"  El NER en control da {res['A']['ner']:.4f} y deberia dar ~{F1_NB11:.4f}.")
    print("  Revisa: columnas _clean, limpiar_rec sin reescribir, ID2LABEL, ruta del modelo.")
else:
    print("="*70)
    print("LECTURA")
    print("="*70)
    print(f"  OK: el control reproduce el nb 11 ({res['A']['ner']:.4f}).")
    print()
    caida_regla = res["A"]["regla"] - res["C"]["regla"]
    caida_ner   = res["A"]["ner"]   - res["C"]["ner"]
    print(f"  1) ¿Falla la regla con un verbo que no conoce?")
    print(f"     Regla: {res['A']['regla']:.4f} -> {res['C']['regla']:.4f}   (caida {caida_regla:+.4f})")
    if caida_regla > 0.5:
        print("     SI. Su cobertura depende de anticipar la redaccion. Era la hipotesis.")
    else:
        print("     NO tanto como se esperaba. Revisar si los reemplazos disparan otro gatillo.")

    print()
    print(f"  2) ¿Aguanta el NER?")
    print(f"     NER: {res['A']['ner']:.4f} -> {res['C']['ner']:.4f}   (caida {caida_ner:+.4f})")

    if caida_ner < 0.10:
        veredicto = "H1 — EL NER GENERALIZA"
        print("     SI. No aprendio una lista de verbos: aprendio que ES una recomendacion.")
        print()
        print(f"  {veredicto}")
        print()
        print("  CONSECUENCIA: el rol de respaldo del NER queda medido sobre")
        print(f"  {len(te)} informes, no sobre los 3 chilenos. Donde la regla cae a")
        print(f"  {res['C']['regla']:.4f}, el NER se mantiene en {res['C']['ner']:.4f}.")
        print()
        print("  Y la diferencia con el verificador de BI-RADS queda nitida:")
        print("     Verificador, sin su señal (el numero)  : 0,939 -> 0,544   colapsa")
        print(f"     NER, sin su señal (el verbo conocido)  : {res['A']['ner']:.3f} -> {res['C']['ner']:.3f}   aguanta")
    else:
        veredicto = "H2 — EL NER TAMBIÉN DEPENDE DEL VERBO"
        print("     NO. Cae junto con la regla.")
        print()
        print(f"  {veredicto}")
        print()
        print("  CONSECUENCIA: el NER aprendio la misma lista de verbos que las reglas,")
        print("  solo que de forma implicita. Su justificacion vuelve a descansar en los")
        print("  3 informes chilenos, y corresponde reportarlo asi.")

    print()
    print(f"  VEREDICTO: {veredicto}")
    print()
    print("  RECORDATORIO: esto es sintetico. Mide el ROL (manejar redacciones no")
    print("  anticipadas), no el desempeño en Chile. Va declarado en el resultado.")

---
## Paso 9 — Guardo

In [ ]:
salida = {
    "experimento": "11c_estres_ner",
    "descripcion": ("Prueba de estres del NER: se perturba el corpus para simular "
                    "redacciones no anticipadas por las reglas (sin encabezado y con "
                    "verbos gatillo fuera de la lista cerrada del modulo) y se compara "
                    "la regla real contra el NER sobre el mismo texto."),
    "motivacion": ("El NER es un respaldo para casos fuera del corpus, asi que la metrica "
                   "del corpus no evalua su rol. Medido dentro del corpus, un regex trivial "
                   "lo empata (0.9991 vs 1.0000). Su justificacion son 3 informes chilenos."),
    "n_test": len(te),
    "reemplazos_usados": REEMPLAZOS,
    "verificacion_reemplazos": "ninguno dispara el regex real _FRASES_GATILLO_RECOMENDACION",
    "condiciones": {k: {"nombre": NOMBRES[k],
                        "f1_regla": float(res[k]["regla"]),
                        "f1_ner":   float(res[k]["ner"])} for k in res},
    "control_valido": bool(CONTROL_OK),
    "veredicto": veredicto,
    "limitacion": ("Sintetico: es el corpus paraguayo perturbado, no informes chilenos "
                   "reales. Mide el rol del componente, no su desempeño en Chile."),
}
Path("resultados").mkdir(exist_ok=True)
with open("resultados/11c_estres_ner.json", "w", encoding="utf8") as f:
    json.dump(salida, f, indent=2, ensure_ascii=False)
print("Guardado en resultados/11c_estres_ner.json\n")
print(json.dumps({"condiciones": salida["condiciones"], "veredicto": veredicto},
                 indent=2, ensure_ascii=False))

---
## Conclusiones

> **Completar tras correr.**

| Condición | Regla | NER |
|---|---|---|
| A · control | `____` | `____` |
| B · sin encabezado | `____` | `____` |
| **C · verbo fuera de la lista** | **`____`** | **`____`** |
| D · ambas | `____` | `____` |

### Si sale H1

> *"El NER es un respaldo para redacciones que las reglas no anticiparon, así que su rol vive fuera del corpus y la métrica del corpus no lo evalúa: ahí un regex lo empata, y lo verifiqué. Como solo tengo tres informes chilenos reales, simulé la brecha sobre los ____ informes del test: reemplacé los verbos gatillo por formas que verifiqué que no están en la lista de reglas. La regla cae de ____ a ____. El NER se mantiene en ____. Eso mide el rol que el componente cumple. Es sintético, y lo declaro: no reemplaza validar con informes chilenos reales, que sigue siendo mi trabajo pendiente."*

### Si sale H2

> *"Le apliqué al NER la prueba de su propio rol y no la pasó: con verbos fuera de la lista, cae junto con las reglas. Aprendió la misma lista, solo que de forma implícita. Su justificación descansa en los tres informes chilenos donde funcionó, y eso es poca evidencia. Lo reporto porque el estándar debe ser el mismo que usé para descartar el verificador."*

### Limitación

Sintético. Corpus paraguayo perturbado, no informes chilenos reales. Mide el rol, no el desempeño en Chile.
